# LLM-as-a-Judge: Bewertung der 240 Experiment-Antworten

Kompakt-Judge gemäß Kapitel 5.1 der Thesis: **ein** Bewertungsaufruf je Antwort erhebt alle drei Metriken (IA, HR, KMC), **ein** Durchgang je Antwort. Evaluator: `claude-opus-4-8`, aufgerufen über die **Claude-Code-CLI im Headless-Modus** (`claude -p`) — die Bewertung läuft damit über dein Claude-Abo, nicht über API-Guthaben. Antwortstruktur per JSON-Schema erzwungen, Metriken werden deterministisch in Python berechnet.

**Voraussetzung:** Die `claude`-CLI ist installiert und mit deinem Abo eingeloggt (einmal `claude` interaktiv starten und anmelden). Ein `ANTHROPIC_API_KEY` wird **nicht** benötigt; falls vorhanden, wird er vom Connector aus der Subprozess-Umgebung entfernt, damit nichts versehentlich per API abgerechnet wird.

**Ablauf mit Gates (Zellen von oben nach unten ausführen):**
1. Setup + `estimate()` (kostenlos)
2. Validierungsstichprobe ziehen (24 Antworten, kostenlos)
3. Handkodierung: `rating_sheet_rater1.csv` (+ `rater2`) mit dem Codebuch ausfüllen
4. Dry Run (kostenlos) → Smoke-Test (1 Bewertung) → Pilot auf der Stichprobe (24 Bewertungen)
5. Übereinstimmung prüfen — **Gate: Krippendorffs α ≥ 0,8 je Metrik**
6. Erst nach bestandenem Gate: Volllauf über alle 240 Antworten
7. Zweitmeinungen (GPT-5.4, Gemini 3 Pro) auf der Stichprobe (API, wenige $)

**Abo-Kontingent statt Dollar:** Die Bewertungen verbrauchen dein Claude-Nutzungskontingent (5-Stunden-Fenster). Wird der Volllauf gedrosselt, einfach später erneut ausführen — die Läufe sind idempotent, fehlende Urteile werden nachgeholt, nichts wird doppelt bewertet.

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
sys.path.append(os.path.abspath(".."))

from dotenv import load_dotenv
load_dotenv("../.env")

from experiment.data_loader import DataLoader
from experiment.judge import ClaudeCLIJudgeConnector, JudgeConnector, JudgeRunner
from experiment.validation_sample import ValidationSampler
from experiment.agreement import AgreementAnalyzer

loader = DataLoader(raw_path="../data/raw", processed_path="../data/processed")

# Offizieller Weg: claude -p über das Abo (Kapitel 5.1 der Thesis).
judge = ClaudeCLIJudgeConnector()
# Alternative über die Anthropic-API (API-Guthaben, benötigt ANTHROPIC_API_KEY):
# judge = JudgeConnector()

runner = JudgeRunner(
    loader=loader,
    connector=judge,
    results_dir="../results",
    key_messages_path="../data/annotations/key_messages.json",
    prices={"input": 0, "output": 0},  # Abo: keine Token-Kosten; Token-Logging bleibt aktiv
)

runner.estimate()  # kostenlose Schätzung des Token-Umfangs (Zeichen/4-Heuristik)

## Schritt 1: Validierungsstichprobe ziehen

Deterministische, geschichtete Ziehung: jede Artikel-Methoden-Zelle genau einmal (8 × 3 = 24), Modelle 12/12 balanciert, Seeds rotierend. Erzeugt `validation_sample.csv`, die leeren Kodiervorlagen `rating_sheet_rater1/2.csv` und je Antwort einen Kodierbogen unter `results/kodierboegen/`.

**Danach Handkodierung:** Kodierbögen + Codebuch (`LaTeX/codebook-answers.pdf`) verwenden und die Zählfelder sowie `KM_i` (ja/nein) in die Kodiervorlage übertragen. Beide Rater kodieren unabhängig voneinander: Rater 1 ist der Autor; Rater 2 ist eine unabhängige, nicht am Untersuchungsdesign beteiligte Person und kodiert ohne Kenntnis der Judge-Urteile und der Kodierung von Rater 1.

**Achtung:** `draw()` schreibt die Kodiervorlagen neu — nach erfolgter Handkodierung diese Zelle **nicht erneut ausführen**, sonst werden die ausgefüllten Bögen überschrieben.

In [2]:
sampler = ValidationSampler(results_dir="../results",
                            key_messages_path="../data/annotations/key_messages.json")
sample = sampler.draw()

NameError: name 'ValidationSampler' is not defined

## Schritt 2: Dry Run (keine Kosten), Smoke-Test (1 Bewertung), Pilot (24 Bewertungen)

Der Dry Run prüft die komplette Pipeline ohne Modellaufrufe. Der Smoke-Test führt **genau eine** echte Bewertung über `claude -p` aus — prüfe danach in `judge_smoke.csv`: gefüllte Token-Spalten, protokollierte `Judge_Model_Version`, plausibles Urteil in `Judge_JSON` (und in der Anthropic-Console, dass **kein** API-Verbrauch entstanden ist). Erst dann den Pilot starten → `judge_pilot.csv`.

In [6]:
runner.run(dry_run=True)  # kostenlos, keine Modellaufrufe

NameError: name 'runner' is not defined

In [ ]:
# Smoke-Test: genau EINE Bewertung über claude -p
first = runner.load_answers()[0]
smoke_key = (first["Article_ID"], first["Method"], first["Model"], first["Seed"])
runner.run(keys={smoke_key}, output_name="judge_smoke", force=True)

In [ ]:
runner.run_sample()  # ECHTLAUF auf den 24 Stichproben-Antworten (Abo-Kontingent)

## Schritt 3: Übereinstimmung prüfen — Gate α ≥ 0,8

Vergleicht die Judge-Urteile (`judge_pilot.csv`) mit den beiden unabhängigen Handkodierungen. Goldstandard ist der Mittelwert beider Rater je Bewertungseinheit; berichtet werden zusätzlich alle paarweisen α (Judge–R1, Judge–R2 sowie R1–R2 als Mensch-Mensch-Referenz). Stammt die gesamte Varianz einer Metrik aus höchstens einer Einheit, ist α wegen des Prävalenzparadoxons nicht aussagekräftig; das Gate wird dann über die exakte Übereinstimmung (≥ 0,95) entschieden und der Fall als degeneriert ausgewiesen. Fehlt `rating_sheet_rater2.csv`, rechnet der Analyzer aus Robustheitsgründen mit Rater 1 allein. **Erst wenn alle drei Metriken das Gate bestehen, den Volllauf starten** — andernfalls Judge-Prompt/Codebuch überarbeiten und den Pilot mit `force=True` wiederholen.

In [ ]:
analyzer = AgreementAnalyzer(results_dir="../results")
report = analyzer.report(
    judge_file="judge_pilot.csv",
    rater1_file="rating_sheet_rater1.csv",
    rater2_file="rating_sheet_rater2.csv",
)

## Schritt 4: Volllauf über alle 240 Antworten (Abo-Kontingent)

**Nur nach bestandenem Gate ausführen.** Idempotent: bereits bewertete Antworten werden übersprungen, Fehler landen in `judge_results_errors.csv` und werden durch erneutes Ausführen nachgeholt — auch nach einer Abo-Drosselung einfach später erneut starten. Ergebnis: `judge_results.csv` (Schema siehe Thesis-Anhang, Tabelle judge-schema).

In [ ]:
runner.run()  # ECHTLAUF: alle 240 Antworten

In [ ]:
runner.status()

## Schritt 5: Zweitmeinungen (Robustheitsindikator, API, wenige $)

Symmetrische Nachbewertung der 24 Stichproben-Antworten durch je ein Modell der GPT- und der Gemini-Familie (Kapitel 5.1). Derselbe Bewertungsprompt läuft über den vorhandenen `LLMConnector` (OpenAI-/Google-API-Keys nötig); das JSON-Format wird per Prompt angefordert und beim Einlesen validiert. Modell-IDs vor dem Lauf prüfen/anpassen.

In [ ]:
from experiment.llm_connector import LLMConnector

gpt_judge = LLMConnector(provider="chatgpt", model_name="gpt-5.4")          # GPT-5.4 (Vollmodell)
gemini_judge = LLMConnector(provider="gemini", model_name="gemini-3-pro")   # Gemini 3 Pro

runner.run_sample(connector=gpt_judge, judge_label="gpt-5.4")        # -> judge_pilot_gpt-5.4.csv
runner.run_sample(connector=gemini_judge, judge_label="gemini-3-pro")  # -> judge_pilot_gemini-3-pro.csv

## Schritt 6: Statistische Auswertung (kostenlos, lokal)

Wertet `judge_results.csv` nach dem in Kapitel 5.1 **vorab festgeschriebenen** Verfahren aus — erst ausführen, wenn alle 240 Urteile vorliegen (das Vollständigkeits-Gate bricht sonst mit der Liste der fehlenden Schlüssel ab):

- **RQ1:** je Metrik × Modell × Methode die gepaarten Differenzen zur Baseline `Original` (n = 15 über Artikel × Seed); zweiseitiger Wilcoxon-Vorzeichen-Rang-Test, Holm-Korrektur über die 7 Methodenvergleiche je Metrik×Modell, Matched-Pairs-rangbiseriale Korrelation (Kerby), Median und Interquartilsabstand. Nulldifferenzen werden ausgeschlossen; besteht eine Bedingung nur aus Nulldifferenzen, wird deskriptiv „kein messbarer Effekt" berichtet.
- **RQ2:** Cross-Model Consistency — bindungskorrigierte Spearman-ρ der Effektvektoren beider Modelle über die 21 Methoden-Artikel-Bedingungen, Seeds zuvor gemittelt.

Die Export-Zelle schreibt CSVs nach `results/`, booktabs-Tabellenfragmente nach `LaTeX/tables/` und Vektor-Abbildungen (PDF) nach `LaTeX/images/`.

In [ ]:
from experiment.analysis import ResultsAnalyzer

stats = ResultsAnalyzer(results_dir="../results")
analysis = stats.report()

In [ ]:
stats.export_csv(analysis)                          # -> results/analysis_rq1|rq2|deskriptiv.csv
stats.export_latex(analysis, "../../LaTeX/tables")  # -> \input-fähige booktabs-Fragmente
stats.export_figures(analysis, "../../LaTeX/images")  # -> Vektor-PDFs für Kapitel 5.2/5.3